In [1]:
from io import StringIO
import bs4
import pandas as pd
import plotly.express as px
import requests as rq
import os

/Users/rachelzawacki/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
page = rq.get(url, headers={"User-Agent": "Mozilla/5.0"})
gdp_soup = bs4.BeautifulSoup(page.text, "html.parser")
tables = gdp_soup.find("table", {"class": "wikitable"})
df = pd.read_html(StringIO(str(tables)))[0]
df.head()
country_col = df.columns[0]

df[country_col] = (df[country_col].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip())
df["Final_GDP"] = pd.to_numeric(df.iloc[:, 1].fillna(df.iloc[:, 2]).fillna(df.iloc[:, 3]).astype(str).str.replace(r"[^\d.]", "", regex=True))

In [6]:
region_url = "https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467"
region_page = rq.get(region_url, headers={"User-Agent": "Mozilla/5.0"})
region_df = pd.read_html(StringIO(region_page.text), match="Country/Territory")[0]

region_countries = (region_df.iloc[:, 0].astype(str).str.strip())
region_names = region_df.iloc[:, 1].astype(str).str.strip()
region_map = dict(zip(region_countries, region_names))

df["Region"] = df[country_col].map(region_map)
df = df.dropna(subset=["Final_GDP", "Region"])
df = df[~df[country_col].str.contains("World|Total|Gross regional product", case=False, na=False)]

fig = px.bar(
    df,
    x="Region",
    y="Final_GDP",
    color=country_col,
    title="GDP by Region",
    labels={"Final_GDP": "GDP (Millions $)", country_col: "Country"},
    template="plotly_white",
)

fig.update_layout(
    barmode="stack",
    xaxis={"categoryorder": "total descending"},
    showlegend=False,
)

output_path = os.path.join("/tmp", "gdp_stacked_bar.html")
fig.write_html(output_path)

fig.show()

In [4]:
fig = px.bar(
    df,
    x="Region",
    y="Final_GDP",
    color=country_col,
    title="Nominal GDP by Region (Stacked by Country)",
    labels={"Final_GDP": "GDP (USD Millions)", country_col: "Country"},
    template="plotly_white",
)

fig.update_layout(
    barmode="stack",
    xaxis={"categoryorder": "total descending"},
    showlegend=False,
)

output_path = os.path.join("/tmp", "gdp_stacked_bar.html")
fig.write_html(output_path)
print(f"Successfully generated plot at {output_path}")

fig.show()

Successfully generated plot at /tmp/gdp_stacked_bar.html
